THIS IS THE FIRST STEP , THE DATA EXPLORATION PHASE, TO HELP US UNDERSTAND THE DATA,COLUMNS, ROWS AND ITS SHAPES

In [1]:
import pandas as pd

#importing the csv file into a pandas dataframe
file_path = "../data/raw/Movies_Genre_Description.csv"
df = pd.read_csv(file_path)

df.head()

,TITLE,GENRE,DESCRIPTION,DATE
0,Oscar et la dame rose,Oscar et la dame rose,Listening in to a conversation between his doc...,2009
1,Cupid,thriller,A brother and sister with a past incestuous re...,1997
2,"Young, Wild and Wonderful",adult,As the bus empties the students for their fiel...,1980
3,The Secret Sin,drama,To help their unemployed father make ends meet...,1915
4,The Unrecovered,drama,The film's title refers not only to the un-rec...,2007


In [3]:
#checking the shape of the dataframe(rows, columns)
number_of_rows, number_of_columns = df.shape

print("Rows:", number_of_rows)
print("Columns:", number_of_columns)

Rows: 108414
Columns: 4


In [4]:
#checking the column names and data types
print("Column Names:")
print(df.columns.tolist())
print()
print("Data Types:")
print(df.dtypes)

Column Names:
['TITLE', 'GENRE', 'DESCRIPTION', 'DATE']

Data Types:
TITLE          str
GENRE          str
DESCRIPTION    str
DATE           str
dtype: object


In [5]:
#checking for missing values in the dataframe
print("Missing Values:")
print(df.isna().sum())

Missing Values:
TITLE          0
GENRE          0
DESCRIPTION    0
DATE           0
dtype: int64


In [6]:
#Inspecting how genres are distributed in the dataset
genre_counts = df["GENRE"].value_counts()

print("Number of genre values:", df["GENRE"].nunique())
print(genre_counts)

Number of genre values: 28
GENRE
drama                    27224
documentary              26192
comedy                   14893
short                    10145
horror                    4408
thriller                  3181
action                    2629
western                   2064
reality-tv                1767
family                    1567
adventure                 1550
music                     1462
romance                   1344
sci-fi                    1293
adult                     1180
crime                     1010
animation                  996
sport                      863
talk-show                  782
fantasy                    645
mystery                    637
musical                    553
biography                  529
history                    486
game-show                  387
news                       362
war                        264
Oscar et la dame rose        1
Name: count, dtype: int64


In [7]:
#inspecting the rare genres in the dataset
genre_counts.tail(10)

GENRE
talk-show                782
fantasy                  645
mystery                  637
musical                  553
biography                529
history                  486
game-show                387
news                     362
war                      264
Oscar et la dame rose      1
Name: count, dtype: int64

In [8]:
#check duplicates
print("Duplicate rows:", df.duplicated().sum())
print(
    "Duplicate descriptions:",
    df["DESCRIPTION"].duplicated().sum()
)

Duplicate rows: 19
Duplicate descriptions: 347


NEXT STEP IS CLEANING THE DATA

In [10]:
#Creating a new dataframe with cleaned data
clean_df = df.copy()
print(clean_df.shape)

(108414, 4)


In [12]:
#Removing leading and trailing whitespace from the DESCRIPTION and GENRE columns
clean_df["DESCRIPTION"] = (
    clean_df["DESCRIPTION"]
    .astype("string")
    .str.strip()
)

clean_df["GENRE"] = (
    clean_df["GENRE"]
    .astype("string")
    .str.strip()
)

print(clean_df.shape)

(108414, 4)


In [14]:
#removing the invalid oscar genre from the dataset
invalid_genre = "Oscar et la dame rose"

clean_df = clean_df[
    clean_df["GENRE"] != invalid_genre
]

print(clean_df.shape)
print(clean_df["GENRE"].nunique())

(108413, 4)
27


In [19]:
#checking the different genres with the exact same description

genre_count_per_description = (
    clean_df.groupby("DESCRIPTION")["GENRE"].nunique()
)

conflicting_descriptions = genre_count_per_description[
    genre_count_per_description > 1
].index

print("different genres with the exact same description:", len(conflicting_descriptions))


different genres with the exact same description: 0


In [17]:
#removing the conflicting descriptions from the dataset
clean_df = clean_df[
    ~clean_df["DESCRIPTION"].isin(conflicting_descriptions)
]
print(clean_df.shape)

(108262, 4)


In [ ]:
#removing duplicate descriptions from the dataset and only keeping the first occurrence of each description
print(clean_df["DESCRIPTION"].duplicated().sum())


231


In [21]:
clean_df = clean_df.drop_duplicates(
    subset="DESCRIPTION",
    keep="first"
)

clean_df = clean_df.reset_index(drop=True)

In [22]:
#verifying that the dataset is clean and ready for model training
print("Cleaned shape:", clean_df.shape)
print("Number of genres:", clean_df["GENRE"].nunique())
print("Missing descriptions:", clean_df["DESCRIPTION"].isna().sum())
print("Missing genres:", clean_df["GENRE"].isna().sum())
print(
    "Duplicate descriptions:",
    clean_df["DESCRIPTION"].duplicated().sum()
)

Cleaned shape: (108031, 4)
Number of genres: 27
Missing descriptions: 0
Missing genres: 0
Duplicate descriptions: 0


DEFINING X AND Y AND SPLITTING THE DATA

In [23]:
#Defining the features and target variable for the model
X = clean_df["DESCRIPTION"]
y = clean_df["GENRE"]

print(X.head())
print()
print(y.head())

print("X shape:", X.shape)
print("y shape:", y.shape)

0    A brother and sister with a past incestuous re...
1    As the bus empties the students for their fiel...
2    To help their unemployed father make ends meet...
3    The film's title refers not only to the un-rec...
4    Quality Control consists of a series of 16mm s...
Name: DESCRIPTION, dtype: string

0       thriller
1          adult
2          drama
3          drama
4    documentary
Name: GENRE, dtype: string
X shape: (108031,)
y shape: (108031,)


In [24]:
#splitting the dataset using scikit
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training descriptions:", X_train.shape)
print("Training labels:", y_train.shape)
print("Testing descriptions:", X_test.shape)
print("Testing labels:", y_test.shape)

Training descriptions: (86424,)
Training labels: (86424,)
Testing descriptions: (21607,)
Testing labels: (21607,)


In [25]:
print("Overall genre proportions:")
print(y.value_counts(normalize=True).head())

print("\nTraining genre proportions:")
print(y_train.value_counts(normalize=True).head())

print("\nTesting genre proportions:")
print(y_test.value_counts(normalize=True).head())

Overall genre proportions:
GENRE
drama          0.250798
documentary    0.242005
comedy         0.137405
short          0.093594
horror         0.040775
Name: proportion, dtype: Float64

Training genre proportions:
GENRE
drama          0.250798
documentary    0.242005
comedy         0.137404
short          0.093597
horror         0.040776
Name: proportion, dtype: Float64

Testing genre proportions:
GENRE
drama          0.250798
documentary    0.242005
comedy         0.137409
short          0.093581
horror         0.040774
Name: proportion, dtype: Float64


Convert descriptions into numbers with TF-IDF (Term Frequency–Inverse Document Frequency)
TFDIF gives gives informative words higher weights and extremely common words lower weights.

In [26]:
#Convert text into numerical vetors using TF-IDF vectorization

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)


Training matrix: (86424, 50000)
Testing matrix: (21607, 50000)


In [27]:
#inspecting the learned vocabulary of the TF-IDF vectorizer
feature_names = tfidf.get_feature_names_out()

print("Vocabulary size:", len(feature_names))
print(feature_names[:20])

Vocabulary size: 50000
['00' '000' '000km' '006' '007' '00am' '00pm' '01' '02' '03' '04' '05'
 '06' '07' '08' '09' '10' '100' '1000' '100th']


                | ghost | love | spaceship |
| Description 1 | 0.8   | 0.0   | 0.4 |
| Description 2 | 0.0   | 1.0   | 0.0 |

In [28]:
#search for the word ghost in the learned vocabulary
word = "ghost"

if word in tfidf.vocabulary_:
    print(f"'{word}' is feature number:", tfidf.vocabulary_[word])
else:
    print(f"'{word}' is not in the vocabulary")

'ghost' is feature number: 18235


In [29]:
example_number = 0

print("Original description:")
print(X_train.iloc[example_number])

example_vector = X_train_tfidf[example_number]

word_indexes = example_vector.indices
word_weights = example_vector.data

example_features = list(
    zip(feature_names[word_indexes], word_weights)
)

example_features = sorted(
    example_features,
    key=lambda item: item[1],
    reverse=True
)

print("\nHighest-weight words:")
print(example_features[:10])

Original description:
The film's main female character has a long history as a psychiatric patient since a very young age. Years of being bullied as a child and during her early teen years has resulted in a distorted self image. To escape from the pain she cuts herself and suffers from eating disorders. We meet her wandering aimlessly and confused along a road. She is picked up by the police and committed yet again to a psychiatric hospital. Ten years of treatment, and the compassion and kindness of one person in particular, finally leads to her healing.

Highest-weight words:
[('psychiatric', np.float64(0.3455468283451961)), ('years', np.float64(0.2261749364565108)), ('aimlessly', np.float64(0.21329473363710075)), ('disorders', np.float64(0.20098650007892513)), ('distorted', np.float64(0.19850888624592866)), ('bullied', np.float64(0.18339773569762788)), ('resulted', np.float64(0.1806448354418231)), ('kindness', np.float64(0.1795310623939609)), ('cuts', np.float64(0.17461305231668378))

Train Multinomial Naive Bayes

In [30]:
#importing and training a Multinomial Naive Bayes model
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()

model.fit(X_train_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](27,)","[2101., 943.,1237.,...,2541., 211.,1649.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](27,)","[-3.72,-4.52,-4.25,...,-3.53,-6.02,-3.96]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U11](27,)","['action','adult','adventure',...,'thriller','war','western']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](27, 50000)","[[0. ,2.49,0. ,...,0.3 ,0. ,0. ], [0. ,0.57,0. ,...,0. ,0. ,0. ], [0.25,1.47,0. ,...,0. ,0. ,0. ], ..., [0. ,1.28,0. ,...,0. ,0. ,0. ], [0. ,0.56,0. ,...,0. ,0. ,0. ], [0. ,2.88,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](27, 50000)","[[-11.03, -9.78,-11.03,...,-10.76,-11.03,-11.03], [-10.92,-10.47,-10.92,...,-10.92,-10.92,-10.92], [-10.72,-10.04,-10.95,...,-10.95,-10.95,-10.95], ..., [-11.07,-10.24,-11.07,...,-11.07,-11.07,-11.07], [-10.84,-10.4 ,-10.84,...,-10.84,-10.84,-10.84], [-10.99, -9.63,-10.99,...,-10.99,-10.99,-10.99]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,50000


In [31]:
# Making predictions on the test set
y_pred = model.predict(X_test_tfidf)

print("Number of predictions:", len(y_pred))
print("First 10 predictions:", y_pred[:10])

Number of predictions: 21607
First 10 predictions: ['documentary' 'drama' 'documentary' 'documentary' 'drama' 'documentary'
 'drama' 'documentary' 'drama' 'drama']


In [32]:
#Inspecting the predictions for a specific example in the test set
example_number = 0

print("Description:")
print(X_test.iloc[example_number])

print("\nActual genre:")
print(y_test.iloc[example_number])

print("\nPredicted genre:")
print(y_pred[example_number])

Description:
Its purpose is to recognize and to honor our Korean American community Heroes and Legends. "Heroes" are ordinary persons who provided extraordinary contributions and services to our Korean American community. "Legends" are the ones with significant professional achievements that made a tremendous impact on our society, community, and peace.

Actual genre:
documentary

Predicted genre:
documentary


In [34]:
#creating new description for prediction
new_description = (
    "A masked killer stalks a group of teenagers "
    "in an abandoned house."
)
new_description_tfidf = tfidf.transform(
    [new_description]
)

new_prediction = model.predict(
    new_description_tfidf
)

print("Predicted genre:", new_prediction[0])


#examine Examine the top three genre probabilities
probabilities = model.predict_proba(
    new_description_tfidf
)[0]

top_three_indexes = probabilities.argsort()[-3:][::-1]

for index in top_three_indexes:
    genre = model.classes_[index]
    probability = probabilities[index]

    print(genre, round(probability, 4))

Predicted genre: horror
horror 0.5302
drama 0.2393
documentary 0.0788


MODEL EVALUATION

In [ ]:
#Importing evaluation metrics from scikit-learn
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

#percentage of correct predictions from the test dataset
accuracy = accuracy_score(y_test, y_pred)

#When the model predicts a particular genre, how often is that prediction correct?
precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

#Out of all movies that really belong to a genre, how many did the model find?
recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

#combines precision and recall into one score
f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:", round(accuracy, 4))
print("Weighted precision:", round(precision, 4))
print("Weighted recall:", round(recall, 4))
print("Weighted F1-score:", round(f1, 4))

Accuracy: 0.482
Weighted precision: 0.4929
Weighted recall: 0.482
Weighted F1-score: 0.3722


In [ ]:
#Support is the number of actual examples of each genre in the test set.
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

              precision    recall  f1-score   support

      action       0.75      0.01      0.01       525
       adult       0.00      0.00      0.00       236
   adventure       0.90      0.03      0.06       309
   animation       0.00      0.00      0.00       198
   biography       0.00      0.00      0.00       106
      comedy       0.57      0.26      0.36      2969
       crime       0.00      0.00      0.00       201
 documentary       0.55      0.90      0.68      5229
       drama       0.41      0.87      0.56      5419
      family       0.00      0.00      0.00       309
     fantasy       0.00      0.00      0.00       128
   game-show       0.00      0.00      0.00        77
     history       0.00      0.00      0.00        96
      horror       0.91      0.05      0.09       881
       music       0.00      0.00      0.00       287
     musical       0.00      0.00      0.00       109
     mystery       0.00      0.00      0.00       127
        news       0.00    